## RAG Pipeline- Data Ingestion to Vector DB

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [5]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: 2-Cloud Computing.pdf
  ✓ Loaded 53 pages

Total documents loaded: 53


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'Office to PDF Converter, PDF-XChange Core API SDK (6.0.319)', 'creator': 'PDF-XChange Editor 6.0.319', 'creationdate': '2024-09-22T15:10:37+05:30', 'moddate': '2024-09-22T15:10:50+05:30', 'title': '02 Cloud Computing.pptx', 'source': '../data/pdf/2-Cloud Computing.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1', 'source_file': '2-Cloud Computing.pdf', 'file_type': 'pdf'}, page_content='CLOUD COMPUTING\nDr. Emmanuel S. Pilli\nMNIT Jaipur'),
 Document(metadata={'producer': 'Office to PDF Converter, PDF-XChange Core API SDK (6.0.319)', 'creator': 'PDF-XChange Editor 6.0.319', 'creationdate': '2024-09-22T15:10:37+05:30', 'moddate': '2024-09-22T15:10:50+05:30', 'title': '02 Cloud Computing.pptx', 'source': '../data/pdf/2-Cloud Computing.pdf', 'total_pages': 53, 'page': 1, 'page_label': '2', 'source_file': '2-Cloud Computing.pdf', 'file_type': 'pdf'}, page_content='Evolution'),
 Document(metadata={'producer': 'Office to PDF Converter, PDF-XChange Core A

In [8]:
### Text Splitting to get chunks

from re import split


def split_documents(documents,chunk_size=1000,chunk_overlap=200):
  """Split doc into smaller chunks"""
  text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=['/n\n','\n',' ','']
  )

  split_docs=text_splitter.split_documents(documents)
  print(f"Split Documents {len(documents)} into {len(split_docs)} chunks.")

  if split_docs:
    print(f"Example Chunk: ")
    print(f"Content: {split_docs[0].page_content[:200]}")
    print(f"Meta Data: {split_docs[0].metadata}")

  return split_docs
    

In [9]:
chunks=split_documents(all_pdf_documents)
print(chunks)

Split Documents 53 into 52 chunks.
Example Chunk: 
Content: CLOUD COMPUTING
Dr. Emmanuel S. Pilli
MNIT Jaipur
Meta Data: {'producer': 'Office to PDF Converter, PDF-XChange Core API SDK (6.0.319)', 'creator': 'PDF-XChange Editor 6.0.319', 'creationdate': '2024-09-22T15:10:37+05:30', 'moddate': '2024-09-22T15:10:50+05:30', 'title': '02 Cloud Computing.pptx', 'source': '../data/pdf/2-Cloud Computing.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1', 'source_file': '2-Cloud Computing.pdf', 'file_type': 'pdf'}
[Document(metadata={'producer': 'Office to PDF Converter, PDF-XChange Core API SDK (6.0.319)', 'creator': 'PDF-XChange Editor 6.0.319', 'creationdate': '2024-09-22T15:10:37+05:30', 'moddate': '2024-09-22T15:10:50+05:30', 'title': '02 Cloud Computing.pptx', 'source': '../data/pdf/2-Cloud Computing.pdf', 'total_pages': 53, 'page': 0, 'page_label': '1', 'source_file': '2-Cloud Computing.pdf', 'file_type': 'pdf'}, page_content='CLOUD COMPUTING\nDr. Emmanuel S. Pilli\nMNIT Jaipur'), Do

## Embedding and Vector Store DB


In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7747.87it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/yy/tj1sngxn6bl0hsj8kzcz02v40000gn/T/ipykernel_8890/2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
